In [1]:
gamer_tag_to_player_id = {
    "BonkCushy": 3475,
    "cruor": 26515,
}

variables = {
    "playerId1": gamer_tag_to_player_id["BonkCushy"],
    "playerId2": gamer_tag_to_player_id["cruor"],
}

In [2]:
%%bash
source ~/.bashrc

In [3]:
import os
import polars as pl
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)

In [4]:
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport
from graphql import get_introspection_query

introspection = client.execute(gql(get_introspection_query()))
json.dump(introspection, open('schema.json', 'w'), indent=2)

In [5]:
# Their gql is borkten, so it just returns all sets for player 2
# just filter it after
query = gql("""
query Sets($playerId1: ID!, $playerId2: ID!) {
  player(id: $playerId1) {
    sets(page: 1, perPage: 200, filters: {playerIds: [$playerId2]}) {
      nodes {
        id
        displayScore
        slots {
          entrant {
            participants {
              player {
                id
                gamerTag
              }
            }
          }
        }
      }
    }
  }
}
""")

result = client.execute(query, variable_values=variables)
# filter results for player 1
result = [s for s in result['player']['sets']['nodes'] if any(
    p['player']['id'] == str(variables['playerId1'])
    for p in s['slots'][0]['entrant']['participants']
)]
result

TransportQueryError: {'message': 'Your query complexity is too high. A maximum of 1000 objects may be returned by each request. (actual: 1811)'}

# grab id from slug

In [ ]:
slug = "user/1f36aaa0"  # cruor
slug = "user/f6746266"  # BonkCushy

query = gql("""
query User($slug: String!) {
  user(slug: $slug) {
    id
    player {
        id
        gamerTag
        tournaments {
        
            nodes {
                id
                name
            }
          }
        }
      }
    }
  }
}
""")

variables = {"slug": slug}
result = client.execute(query, variable_values=variables)
result

GraphQLSyntaxError: Syntax Error: Unexpected '}'.

GraphQL request:17:4
16 |     }
17 |   }
   |    ^
18 | }